# Fusion signature deconvolution on TCGA bulk RNA-seq

Applies the fused/parental/resistant scRNA-seq signature (`fusion_signature_expression.h5ad`) to real
TCGA bulk RNA-seq samples via NNLS deconvolution, then compares estimated fused-cell fraction across
organ/tissue types.

Gene expression is streamed from [UCSC Xena](https://xenabrowser.net) with `xenaPython`, querying only
the signature gene panel (not the full ~50k-gene x ~10.5k-sample matrix), so nothing beyond a small
samples x signature-genes table is ever held in memory. `xenaPython` is not installed on the HPC, so an
offline/full-download fallback (plain `requests` + chunked `pandas`) is included at the bottom, commented
out, for running there instead.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import nnls

import xenaPython as xena

## 1. Load the fusion signature reference

Same signature-gene reference used in `Fusion_signature_pseudobulk_deconvolution.ipynb`. Unlike that
notebook, there's no held-out validation split here — TCGA's true fused fraction is exactly what we're
trying to estimate, so we use *all* reference cells to build the best possible per-class signature
matrix ("production" reference rather than a validation split).

In [ ]:
SIGNATURE_EXPRESSION_PATH = 'fusion_signature_expression.h5ad'

sig_adata = ad.read_h5ad(SIGNATURE_EXPRESSION_PATH)
signature_genes = sig_adata.var_names.tolist()
classes = sorted(sig_adata.obs['sample'].unique())
print(f'{len(signature_genes)} signature genes, {sig_adata.n_obs} reference cells')
print('Classes:', classes)

def to_linear(X):
    return np.sinh(X)

expr_asinh = sig_adata.X
if hasattr(expr_asinh, 'toarray'):
    expr_asinh = expr_asinh.toarray()
expr_linear = to_linear(expr_asinh)

sample_labels = sig_adata.obs['sample'].values
reference_matrix = np.column_stack([
    expr_linear[sample_labels == c].mean(axis=0)
    for c in classes
])  # (n_genes, n_classes)
reference_pool_mean = expr_linear.mean(axis=0)  # (n_genes,), all classes pooled - used for cross-platform rescaling below

print('Reference matrix shape:', reference_matrix.shape)

## 2. Pull TCGA sample/tissue metadata from Xena

`tcga_RSEM_gene_tpm` (hub `toilHub`) is the TCGA-only RSEM TPM matrix (log2(TPM+0.001)), 10,535 samples.
Its companion phenotype dataset `TcgaTargetGTEX_phenotype.txt` carries `_primary_site` (organ/tissue),
`_sample_type` (Primary Tumor / Solid Tissue Normal / Metastatic / ...), and `_study`.

Phenotype fields are categorical and come back from the hub as integer codes, decoded via `field_codes`.

In [ ]:
XENA_HOST = xena.PUBLIC_HUBS['toilHub']
EXPR_DATASET = 'tcga_RSEM_gene_tpm'
PHENO_DATASET = 'TcgaTargetGTEX_phenotype.txt'
PHENO_FIELDS = ['_primary_site', '_sample_type', '_study']
SAMPLE_TYPES_KEPT = ['Primary Tumor']  # drop Solid Tissue Normal / Metastatic / etc.
BATCH_SIZE = 1000

def decode_categorical(host, dataset, samples, fields, batch_size=BATCH_SIZE):
    code_lookup = {
        d['name']: d['code'].split('\t')
        for d in xena.field_codes(host, dataset, fields)
    }
    rows = []
    for i in range(0, len(samples), batch_size):
        batch = samples[i:i + batch_size]
        raw = xena.dataset_fetch(host, dataset, batch, fields)  # raw[f_idx][sample_idx] = code
        decoded = {
            field: [code_lookup[field][v] if v >= 0 else None for v in raw[f_idx]]
            for f_idx, field in enumerate(fields)
        }
        rows.append(pd.DataFrame({'sampleID': batch, **decoded}))
    return pd.concat(rows, ignore_index=True)

tcga_samples = xena.dataset_samples(XENA_HOST, EXPR_DATASET, None)
print(f'{len(tcga_samples)} total TCGA samples in {EXPR_DATASET}')

pheno_df = decode_categorical(XENA_HOST, PHENO_DATASET, tcga_samples, PHENO_FIELDS)
pheno_df = pheno_df[pheno_df['_study'] == 'TCGA']
pheno_df = pheno_df[pheno_df['_sample_type'].isin(SAMPLE_TYPES_KEPT)].reset_index(drop=True)

print(f'{len(pheno_df)} samples after filtering to {SAMPLE_TYPES_KEPT}')
print(pheno_df['_primary_site'].value_counts())

## 3. Stream signature-gene expression for the selected samples

Only `signature_genes` are requested per batch via `dataset_gene_probe_avg` — never the full matrix.

In [ ]:
bulk_sample_ids = pheno_df['sampleID'].tolist()

matched_genes = [g for g in signature_genes if g in xena.dataset_field(XENA_HOST, EXPR_DATASET)]
missing_genes = sorted(set(signature_genes) - set(matched_genes))
print(f'{len(matched_genes)}/{len(signature_genes)} signature genes found in {EXPR_DATASET}')
if missing_genes:
    print('Missing (dropped):', missing_genes)

expr_chunks = []
for i in range(0, len(bulk_sample_ids), BATCH_SIZE):
    batch = bulk_sample_ids[i:i + BATCH_SIZE]
    gene_records = xena.dataset_gene_probe_avg(XENA_HOST, EXPR_DATASET, batch, matched_genes)
    chunk = pd.DataFrame(
        {rec['gene']: rec['scores'][0] for rec in gene_records},
        index=batch,
    )
    expr_chunks.append(chunk)
    print(f'  fetched {min(i + BATCH_SIZE, len(bulk_sample_ids))}/{len(bulk_sample_ids)} samples', end='\r')

tcga_log2tpm = pd.concat(expr_chunks)[matched_genes]  # (n_samples, n_matched_genes), log2(TPM + 0.001)
tcga_linear = 2 ** tcga_log2tpm - 0.001
print(f'\nTCGA expression matrix: {tcga_linear.shape}')

## 4. NNLS deconvolution per TCGA sample

Same NNLS approach as the pseudo-bulk notebook: solve `reference_matrix @ fractions ≈ bulk_profile` for
non-negative `fractions`, then normalize to sum to 1.

**Cross-platform caveat:** the reference is built from 10x scRNA-seq (asinh-transformed, per-cell
normalized) and the bulk data is RSEM TPM from a different assay entirely — the two are not on
inherently comparable absolute scales. Any single *per-sample* multiplicative scale difference cancels
out automatically once fractions are re-normalized to sum to 1, but *per-gene* scale differences between
platforms do not. As a lightweight correction, each gene's bulk values are rescaled by the ratio of its
mean in the scRNA reference pool to its mean across the bulk cohort, i.e. a simple mean-matching batch
correction — not a substitute for formal cross-platform normalization (e.g. quantile normalization to a
shared reference, as in CIBERSORTx), but enough to remove the biggest gene-level scale offsets. Treat
resulting fractions as indicative/relative estimates, not calibrated absolute percentages, unless this is
validated further (e.g. against pseudo-bulks built directly from bulk-scale data).

In [ ]:
bulk_pool_mean = tcga_linear.mean(axis=0).values  # (n_matched_genes,)
gene_idx = [signature_genes.index(g) for g in matched_genes]
scale_factor = reference_pool_mean[gene_idx] / np.where(bulk_pool_mean == 0, np.nan, bulk_pool_mean)
scale_factor = np.nan_to_num(scale_factor, nan=1.0)

reference_matrix_matched = reference_matrix[gene_idx, :]

nnls_records = []
for sample_id, row in tcga_linear.iterrows():
    bulk_profile_rescaled = row.values * scale_factor
    coeffs, _residual = nnls(reference_matrix_matched, bulk_profile_rescaled)
    total = coeffs.sum()
    fractions = coeffs / total if total > 0 else np.zeros_like(coeffs)
    nnls_records.append({'sampleID': sample_id, **dict(zip(classes, fractions))})

nnls_df = pd.DataFrame(nnls_records)
tcga_results_df = pheno_df.merge(nnls_df, on='sampleID')
tcga_results_df = tcga_results_df.rename(columns={c: f'{c}_fraction' for c in classes})
tcga_results_df.to_csv('tcga_fusion_deconvolution_results.csv', index=False)
tcga_results_df.head()

## 5. Boxplot: estimated fused fraction by organ/tissue type

In [ ]:
MIN_SAMPLES_PER_ORGAN = 5

organ_counts = tcga_results_df['_primary_site'].value_counts()
kept_organs = organ_counts[organ_counts >= MIN_SAMPLES_PER_ORGAN].index
plot_df = tcga_results_df[tcga_results_df['_primary_site'].isin(kept_organs)]

organ_order = (
    plot_df.groupby('_primary_site')['fused_fraction']
    .median()
    .sort_values(ascending=False)
    .index
)

fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(organ_order)), 6))
sns.boxplot(data=plot_df, x='_primary_site', y='fused_fraction', order=organ_order, ax=ax)
ax.set_xlabel('Organ / tissue type (primary site)')
ax.set_ylabel('Estimated fused-cell fraction (NNLS)')
ax.set_title('TCGA bulk RNA-seq: estimated fused fraction by organ')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

for i, organ in enumerate(organ_order):
    n = organ_counts[organ]
    ax.annotate(f'n={n}', (i, ax.get_ylim()[1]), ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('tcga_fused_fraction_by_organ.png', dpi=150)
plt.show()

---
## Offline / HPC fallback (no `xenaPython`)

`xenaPython` isn't installed on the HPC. The block below is the equivalent full-download-and-subset
path: pulls the raw Xena bulk `.gz` matrix and phenotype file over plain HTTP, reads the (large) matrix
in row chunks so the full ~50k-gene x ~10.5k-sample matrix is never fully materialized, keeps only the
signature-gene rows, and writes intermediate/final outputs to
`/mnt/vstor/SOM_CCCC_JGS/shultesp/data`. Uses `tcga_RSEM_gene_tpm`'s Hugo-symbol sibling dataset
(`tcga_RSEM_Hugo_norm_count`) so rows are already gene symbols and no probeMap/Ensembl-ID join step is
needed.

Left commented out — uncomment to run on the HPC once `signature_genes` is available in that
environment (e.g. by also copying `fusion_signature_expression.h5ad` there, or hardcoding the gene list).

In [ ]:
# import os
# import requests
#
# DATA_DIR = '/mnt/vstor/SOM_CCCC_JGS/shultesp/data'
# os.makedirs(DATA_DIR, exist_ok=True)
#
# EXPR_URL = 'https://toil.xenahubs.net/download/tcga_RSEM_Hugo_norm_count.gz'
# PHENO_URL = 'https://toil.xenahubs.net/download/TcgaTargetGTEX_phenotype.txt.gz'
# EXPR_LOCAL = os.path.join(DATA_DIR, 'tcga_RSEM_Hugo_norm_count.gz')
# PHENO_LOCAL = os.path.join(DATA_DIR, 'TcgaTargetGTEX_phenotype.txt.gz')
#
# def download(url, path):
#     if os.path.exists(path):
#         return
#     with requests.get(url, stream=True) as r:
#         r.raise_for_status()
#         with open(path, 'wb') as f:
#             for chunk in r.iter_content(chunk_size=1 << 20):
#                 f.write(chunk)
#
# download(EXPR_URL, EXPR_LOCAL)
# download(PHENO_URL, PHENO_LOCAL)
#
# pheno_df = pd.read_csv(PHENO_LOCAL, sep='\t')
# pheno_df = pheno_df[pheno_df['_study'] == 'TCGA']
# pheno_df = pheno_df[pheno_df['_sample_type'].isin(SAMPLE_TYPES_KEPT)].reset_index(drop=True)
#
# CHUNKSIZE = 2000  # genes per chunk, not samples - this file is genes-as-rows
# matched_chunks = []
# reader = pd.read_csv(EXPR_LOCAL, sep='\t', index_col=0, chunksize=CHUNKSIZE)
# for chunk in reader:
#     hit = chunk.loc[chunk.index.isin(signature_genes)]
#     if not hit.empty:
#         matched_chunks.append(hit)
#
# tcga_log2norm = pd.concat(matched_chunks).T  # (n_samples, n_matched_genes)
# tcga_log2norm = tcga_log2norm.loc[tcga_log2norm.index.isin(pheno_df['sampleID'])]
# tcga_linear = 2 ** tcga_log2norm - 1
#
# tcga_linear.to_csv(os.path.join(DATA_DIR, 'tcga_signature_gene_expression_linear.csv'))
# pheno_df.to_csv(os.path.join(DATA_DIR, 'tcga_phenotype_filtered.csv'), index=False)
#
# # from here, join tcga_linear with pheno_df and reuse the NNLS + rescaling cell above unchanged